# 01-3. 固有値・固有ベクトル — 動かして確かめる

📖 解説: [`../03_eigenvalues.md`](../03_eigenvalues.md)

## このノートで触るもの
1. 固有値・固有ベクトルの基本計算
2. 「向きが変わらない」を可視化
3. 対角化と行列のべき乗
4. 対称行列のスペクトル分解
5. PCA を NumPy で自作 → 高次元データを2Dに
6. Power Iteration で最大固有値

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (03_eigenvalues.md)](../03_eigenvalues.md)

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib inline
rng = np.random.default_rng(seed=42)

## 1. 基本計算

$$
A \mathbf{v} = \lambda \mathbf{v}
$$

$\lambda$ = 固有値、$\mathbf{v}$ = 固有ベクトル

In [ ]:
A = np.array([[2.0, 1.0],
              [0.0, 3.0]])

eigvals, eigvecs = np.linalg.eig(A)
print('固有値:', eigvals)
print('固有ベクトル (列):')
print(eigvecs)

# 検算: A v = λ v ?
for i in range(len(eigvals)):
    v = eigvecs[:, i]
    lam = eigvals[i]
    print(f'\nλ_{i} = {lam:.4f}, v_{i} = {v}')
    print(f'  A v   = {A @ v}')
    print(f'  λ v   = {lam * v}')
    print(f'  一致: {np.allclose(A @ v, lam * v)}')

## 2. 「向きが変わらない」を可視化

ランダムなベクトル群と、固有ベクトルを変換してみます。
固有ベクトルだけは「向きが変わらず長さだけ変わる」ことを観察してください。

固有ベクトルだけは変換後も**同じ方向**を向く ($\lambda$ 倍に伸縮)

In [ ]:
def visualize_eigenvectors(scale: float) -> None:
    """行列 A による変換を可視化. 普通のベクトルは向きが変わるが、固有ベクトルは変わらない."""
    A = np.array([[2.0, 1.0],
                  [1.0, 2.0]])    # 対称、固有値 1, 3
    eigvals, eigvecs = np.linalg.eigh(A)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # 普通のベクトル (青)
    angles = np.linspace(0, 2 * np.pi, 8, endpoint=False)
    for ang in angles:
        v = scale * np.array([np.cos(ang), np.sin(ang)])
        Av = A @ v
        axes[0].quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='lightblue', alpha=0.6)
        axes[1].quiver(0, 0, Av[0], Av[1], angles='xy', scale_units='xy', scale=1, color='lightblue', alpha=0.6)

    # 固有ベクトル (赤)
    for i in range(2):
        v = scale * eigvecs[:, i]
        Av = A @ v
        axes[0].quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='red', linewidth=2)
        axes[1].quiver(0, 0, Av[0], Av[1], angles='xy', scale_units='xy', scale=1, color='red', linewidth=2)
        axes[0].text(v[0]*1.1, v[1]*1.1, f'v{i}', fontsize=11, color='red')
        axes[1].text(Av[0]*1.05, Av[1]*1.05, f'λ{i}={eigvals[i]:.2f}', fontsize=11, color='red')

    for ax, title in zip(axes, ['変換前 (青=普通, 赤=固有ベクトル)', '変換後: 赤(固有ベクトル) は向きそのまま、青は向きが変わる']):
        ax.set_xlim(-5, 5)
        ax.set_ylim(-5, 5)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

interact(
    visualize_eigenvectors,
    scale=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description='スケール'),
);

## 3. 対角化と行列のべき乗

$$
A = P D P^{-1}, \quad A^n = P D^n P^{-1}
$$

In [ ]:
A = np.array([[2.0, 1.0],
              [0.0, 3.0]])

eigvals, P = np.linalg.eig(A)
D = np.diag(eigvals)
P_inv = np.linalg.inv(P)

print('A の固有値:', eigvals)
print('\nP D P⁻¹ で復元:')
print(P @ D @ P_inv)

# A^10 を 2通りで
print('\nA^10 (普通の積):')
A10_naive = A.copy()
for _ in range(9):
    A10_naive = A10_naive @ A
print(A10_naive)

print('\nA^10 (対角化経由 P D^10 P⁻¹):')
A10_eig = P @ np.diag(eigvals ** 10) @ P_inv
print(A10_eig)

print(f'\n一致: {np.allclose(A10_naive, A10_eig)}')

## 4. 対称行列のスペクトル分解 (eigh の威力)

$$
A = Q \Lambda Q^\top, \quad Q^\top Q = I
$$

対称行列は実固有値・直交固有ベクトルを持つ

In [ ]:
A = np.array([[4.0, 1.0, 2.0],
              [1.0, 5.0, 0.0],
              [2.0, 0.0, 6.0]])
print(f'対称か: {np.allclose(A, A.T)}')

eigvals, Q = np.linalg.eigh(A)    # 対称専用、必ず実数を返す
print(f'\n固有値 (昇順): {eigvals}')

# Q は直交行列: Q^T Q = I
print(f'\nQ^T Q ≈ I:')
print(Q.T @ Q)
print(f'I に近い: {np.allclose(Q.T @ Q, np.eye(3))}')

# 復元 A = Q Λ Q^T
print(f'\nQ Λ Qᵀ で A を復元:')
print(Q @ np.diag(eigvals) @ Q.T)

## 5. PCA を NumPy で実装する → 4次元データを2Dに圧縮

共分散行列の固有値分解:

$$
\Sigma = Q \Lambda Q^\top
$$

大きい固有値の固有ベクトル = データが最も広がる方向

In [ ]:
def pca(X: np.ndarray, n_components: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """主成分分析.
    
    Returns:
        X_pca: 削減後データ (n, n_components)
        components: 主成分 (n_components, d)
        explained_variance: 各主成分の固有値（分散）
    """
    X_centered = X - X.mean(axis=0)
    cov = np.cov(X_centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    # 大きい順にソート
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    components = eigvecs[:, :n_components].T
    X_pca = X_centered @ components.T
    return X_pca, components, eigvals[:n_components]

# 4次元データを生成 (実は2次元的な構造を持つ)
n = 200
true_2d = rng.standard_normal((n, 2)) @ np.array([[3, 0], [0, 1]])  # 横長の楕円
noise = rng.standard_normal((n, 4)) * 0.3
# 2次元データを4次元に埋め込み + ノイズ
M = rng.standard_normal((2, 4))
X = true_2d @ M + noise

print(f'元データ shape: {X.shape}')
X_pca, comps, vars_ = pca(X, n_components=2)
print(f'PCA後 shape:    {X_pca.shape}')
print(f'各主成分の分散 (固有値): {vars_}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X[:, 0], X[:, 1], s=10, alpha=0.5)
axes[0].set_xlabel('特徴量 0')
axes[0].set_ylabel('特徴量 1')
axes[0].set_title('元データ (4次元のうち先頭2軸を見ている)')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], s=10, alpha=0.5, color='orange')
axes[1].set_xlabel('主成分1')
axes[1].set_ylabel('主成分2')
axes[1].set_title('PCA で2次元に圧縮 (本質的な構造が見える)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Power Iteration — Google PageRank の素朴版

最大固有値の固有ベクトルを、行列の繰り返し掛け算で求める方法。
Google が初期に Web 全体の重要度を計算した方法とほぼ同じ。

最大固有値の固有ベクトルを求める反復法:

$$
\mathbf{v}_{k+1} = \frac{A \mathbf{v}_k}{\|A \mathbf{v}_k\|}, \quad \lambda \approx \mathbf{v}^\top A \mathbf{v}
$$

In [ ]:
def power_iteration(A: np.ndarray, n_iter: int = 100) -> tuple[float, np.ndarray]:
    """Power method: 最大固有値とその固有ベクトル."""
    n = A.shape[0]
    v = rng.standard_normal(n)
    v = v / np.linalg.norm(v)
    for _ in range(n_iter):
        v_new = A @ v
        v_new = v_new / np.linalg.norm(v_new)
        v = v_new
    # レイリー商で固有値を取得
    lam = float(v @ A @ v)
    return lam, v

A = np.array([[4.0, 1.0],
              [2.0, 3.0]])
lam, v = power_iteration(A, n_iter=200)
print(f'Power method の最大固有値: {lam:.4f}')
print(f'対応する固有ベクトル:      {v}')

# 検算
true_eigvals = np.linalg.eigvals(A)
print(f'\nNumPy の全固有値: {true_eigvals}    ← 最大が一致するか?')

## まとめ

ここで触ったもの:
- 基本計算と検算 (Av = λv)
- 「向きが変わらない方向」の可視化
- 対角化と行列のべき乗高速化
- 対称行列の eigh
- PCA で次元削減
- Power Iteration

## 次へ

次のノート → [`04_decompositions.ipynb`](04_decompositions.ipynb)

解説 → [`../04_decompositions.md`](../04_decompositions.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`02_matrices.ipynb`](02_matrices.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`04_decompositions.ipynb`](04_decompositions.ipynb) |